# Lessson 3: SingleTaskGP Internals: Kernel, Lengthscale, ARD, Noise, Likelihood, and Learned Parameters

## What exactly is `SingleTaskGP`?

At a high level:

$$
f(x)\sim\mathcal{GP}\left(m(x),k(x,x')\right).
$$

You provide:

$$
D=\{(x_i,y_i)\}_{i=1}^{n}
$$

and the GP learns a probabilistic representation of:

$$
f(x).
$$

The basic architecture is:

```text
SingleTaskGP
│
├── Mean module
│
├── Covariance module
│     │
│     └── Kernel
│
├── Likelihood
│
└── Outcome transform
```

The most important part for today is:

```text
Covariance module
        │
        ▼
      Kernel
        │
        ├── lengthscale
        └── outputscale
```

## Why does a GP need a kernel?

The GP needs to know how similar two input points are.

Suppose:

$$
x_1=0.30
$$

and:

$$
x_2=0.31.
$$

We intuitively expect their objective values to be relatively similar.

But if:

$$
x_1=0.30
$$

and:

$$
x_2=0.95,
$$

we may expect them to be less correlated.

The kernel mathematically expresses this idea:

$$
k(x,x').
$$

It produces a covariance/similarity between two points.

### The RBF kernel intuition

A classic kernel is the RBF kernel:

$$
\boxed{
k(x,x')
=
\sigma_f^2
\exp\left[
-\frac{1}{2}
\frac{(x-x')^2}{\ell^2}
\right]
}
$$

where:

- $x,x'$: two inputs
- $\ell$: lengthscale
- $\sigma_f^2$: output variance

The critical parameter is: $\ell$

#3 What does lengthscale mean?

Imagine:

$$
\ell=0.1.
$$

Then the function can change relatively quickly as $x$ changes.

If:

$$
\ell=10,
$$

the function varies much more slowly.

So:

```text
small lengthscale
      ↓
rapid variation
      ↓
highly local behavior
```

while:

```text
large lengthscale
      ↓
smooth variation
      ↓
global behavior
```

This becomes particularly important for your materials optimization.

### Example: temperature

Suppose your input is normalized temperature:

$$
x_T\in[0,1].
$$

If the fitted GP learns:

$$
\ell_T=0.05,
$$

then relatively small changes in temperature can produce meaningful changes in the predicted objective.

If it learns:

$$
\ell_T=0.8,
$$

the objective appears much smoother with respect to temperature.

So the learned lengthscale tells us something about the **variation scale of the surrogate function**.

## Create a simple GP
Let's build a simple example.

In [3]:
import torch

torch.set_default_dtype(torch.double)

train_X = torch.tensor([
    [0.00],
    [0.20],
    [0.40],
    [0.60],
    [0.80],
    [1.00],
])

train_Y = torch.tensor([
    [0.10],
    [0.85],
    [0.90],
    [0.20],
    [-0.80],
    [-0.10],
])

In [5]:
from botorch.models import SingleTaskGP

model = SingleTaskGP(
    train_X,
    train_Y,
)

print(model)

SingleTaskGP(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): GammaPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): MaternKernel(
      (lengthscale_prior): GammaPrior()
      (raw_lengthscale_constraint): Positive()
    )
    (outputscale_prior): GammaPrior()
    (raw_outputscale_constraint): Positive()
  )
)


###  The mean module

A GP has:

$$
m(x).
$$

This is the prior mean function.

For many standard BO problems, the mean is relatively simple.

You can inspect:

```python
print(model.mean_module)
```

Depending on the model configuration, you will typically see a constant mean component.

Conceptually:

$$
m(x)=c.
$$

The model learns an appropriate baseline level.

---

### The covariance module

Now:

```python
print(model.covar_module)
```

This is much more interesting.

The covariance module determines:

$$
k(x,x').
$$

In the standard `SingleTaskGP`, you will typically encounter a scaled kernel structure.

Conceptually:

```text
ScaleKernel
     │
     ▼
 base kernel
     │
     ▼
 covariance
```

The scale controls the overall magnitude of variation.

---

### Why `ScaleKernel`?

Suppose the base kernel produces:

$$
k_\text{base}(x,x').
$$

The scaled kernel gives:

$$
k(x,x')
=
\sigma_f^2
k_\text{base}(x,x').
$$

So there are two different concepts:

### Lengthscale

Controls **how quickly similarity decreases with distance**.

### Output scale

Controls **how large the function variations are**.

---

### Inspect outputscale

Try:

```python
print(model.covar_module.outputscale)
```

This gives the learned output variance scale.

You can also access:

```python
model.covar_module.outputscale.item()
```

after fitting.

For example, conceptually:

```text
outputscale = 1.37
```

means the GP has learned a certain overall function variance.

---

### Inspect the base kernel

Try:

```python
print(model.covar_module.base_kernel)
```

This lets you inspect the kernel underneath the scaling.

You will typically find parameters such as:

```text
lengthscale
```
Before fitting, this is an initial value.


In [6]:
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.fit import fit_gpytorch_mll

mll = ExactMarginalLogLikelihood(
    model.likelihood,
    model,
)

fit_gpytorch_mll(mll)

ExactMarginalLogLikelihood(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): GammaPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (model): SingleTaskGP(
    (likelihood): GaussianLikelihood(
      (noise_covar): HomoskedasticNoise(
        (noise_prior): GammaPrior()
        (raw_noise_constraint): GreaterThan(1.000E-04)
      )
    )
    (mean_module): ConstantMean()
    (covar_module): ScaleKernel(
      (base_kernel): MaternKernel(
        (lengthscale_prior): GammaPrior()
        (raw_lengthscale_constraint): Positive()
      )
      (outputscale_prior): GammaPrior()
      (raw_outputscale_constraint): Positive()
    )
  )
)

In [8]:
# shows the learned value.

print(
    model.covar_module.base_kernel.lengthscale
)

tensor([[0.3030]], grad_fn=<SoftplusBackward0>)


### What shape is the lengthscale?

For one input dimension, you may see something conceptually like:

```text
[1, 1, 1]
```

or an equivalent shape depending on the underlying kernel structure.

Don't focus only on the printed shape.

The important question is:

> Is there one lengthscale or one lengthscale per input dimension?

That leads us to **ARD**.

---

### ARD — Automatic Relevance Determination

Suppose:

$$
x=[x_1,x_2,x_3].
$$

A GP could use one common lengthscale:

$$
\ell.
$$

Or it can learn:

$$
\ell_1,\ell_2,\ell_3.
$$

The latter is **ARD**:

$$
\boxed{
\text{Automatic Relevance Determination}
}
$$

The kernel becomes:

$$
k(x,x')
=
\sigma_f^2
\exp\left[
-\frac12
\sum_{j=1}^{d}
\frac{(x_j-x_j')^2}{\ell_j^2}
\right].
$$

---

### Why ARD is important for materials science

Suppose your variables are:

```text
x1 = salt concentration
x2 = EC fraction
x3 = DMC fraction
x4 = temperature
x5 = mixing time
```

The GP could learn:

$$
\ell=
[0.08,\ 0.50,\ 0.65,\ 0.12,\ 0.80].
$$

This suggests:

```text
small lengthscale
→ objective changes rapidly
→ strong local variation

large lengthscale
→ objective changes slowly
→ smoother dependence
```

Therefore:

```text
salt concentration  → sensitive
temperature         → sensitive

EC fraction         → smoother
DMC fraction        → smoother
mixing time         → smoother
```

This is potentially scientifically useful.

But be careful:

> A small learned lengthscale is **not automatically proof of physical importance or causality**.

It is a property of the fitted surrogate under the given data and kernel assumptions.

---

### How ARD appears in BoTorch

A typical kernel configuration has:

```python
ard_num_dims=d
```

For example:

```python
ard_num_dims=5
```

means:

$$
\ell_1,\ell_2,\ell_3,\ell_4,\ell_5.
$$

This is why the input dimensionality matters.

If:

```python
train_X.shape == [50, 5]
```

then:

$$
d=5.
$$

---

### A critical warning about interpreting lengthscales

Suppose:

$$
\ell_1=0.05
$$

and:

$$
\ell_2=0.5.
$$

You should **not** compare these directly if the original variables were on radically different scales.

That's another reason input normalization matters.

For example:

```text
temperature: 700–900 °C
salt:        0.5–2.0 M
```

must be transformed consistently before interpreting GP lengthscales.

Otherwise, the numerical magnitude of the lengthscale partly reflects the units.

## The likelihood

Now inspect:

```python
print(model.likelihood)
```

The likelihood describes the observation process.

The basic noisy observation model is:

$$
y=f(x)+\epsilon
$$

where:

$$
\epsilon\sim\mathcal{N}(0,\sigma_n^2).
$$

Therefore:

$$
y\sim\mathcal{N}\left(f(x),\sigma_n^2\right).
$$

### Signal vs noise

This is fundamental.

Suppose the true function is:

$$
f(x).
$$

The SDL measurement is:

$$
y=f(x)+\epsilon.
$$

The GP therefore needs to distinguish:

```text
signal
  ↓
real variation in the underlying material behavior

noise
  ↓
measurement / experimental variability
```

The likelihood controls this observation noise.


### Inspect GP noise

For a fitted model, you can inspect the likelihood's noise parameter.

For example:

```python
print(model.likelihood.noise)
```

or:

```python
print(
    model.likelihood.noise.item()
)
```

The exact representation depends on the likelihood configuration.

Conceptually, if:

$$
\sigma_n^2=0.01,
$$

the model believes observations have approximately that amount of observation variance.

### Why noise matters in your SDL

Imagine the SDL evaluates:

```text
electrolyte A:
1.51
1.49
1.52
```

versus:

```text
electrolyte B:
1.50
1.20
1.80
```

The second experiment is much noisier.

A GP that understands observation noise shouldn't interpret every variation as real chemistry.

This becomes particularly important when:

- electrochemical measurements drift
- cell-to-cell variation exists
- fabrication has stochasticity
- measurement equipment has noise
- repeated experiments disagree

## Fitting means learning these parameters

When you run:

```python
fit_gpytorch_mll(mll)
```

BoTorch/GPyTorch optimizes the model parameters.

Conceptually:

```text
training data
     │
     ▼
marginal likelihood
     │
     ▼
optimization
     │
     ├── mean parameters
     ├── lengthscale
     ├── outputscale
     └── noise
     │
     ▼
fitted GP
```

### The marginal likelihood

For an exact GP:

$$
\mathbf{y}
\sim
\mathcal{N}
\left(
\mathbf{m},
K+\sigma_n^2I
\right).
$$

The log marginal likelihood is:

$$
\log p(\mathbf{y}\mid X,\theta).
$$

Expanding conceptually:

$$
\log p(\mathbf{y}\mid X,\theta)
=
-\frac12
(\mathbf{y}-\mathbf{m})^T
(K+\sigma_n^2I)^{-1}
(\mathbf{y}-\mathbf{m})
$$

$$
-\frac12
\log\left|K+\sigma_n^2I\right|
-\frac{n}{2}\log(2\pi).
$$

The optimizer adjusts:

$$
\theta
$$

to make the observed data plausible under the GP.


### Why isn't the GP just interpolating everything?

Because the likelihood includes noise and the kernel has finite flexibility.

Suppose two observations are:

```text
x=0.50 → 1.00
x=0.51 → 1.40
```

A GP doesn't necessarily conclude:

> "The function must jump sharply."

It considers:

- kernel smoothness
- noise
- other observations
- overall marginal likelihood.

This is one of the strengths of probabilistic surrogate modeling.


### Inspect all learnable parameters

A very useful debugging technique is:

```python
for name, param in model.named_parameters():
    print(name)
    print(param)
```

After fitting:

```python
for name, param in model.named_parameters():
    print(
        name,
        param.detach()
    )
```

You will see parameter names corresponding to components such as:

```text
mean_module.raw_constant
covar_module.raw_outputscale
covar_module.base_kernel.raw_lengthscale
likelihood.noise_covar.raw_noise
```

The exact names can vary by model/kernel version.



### Why does it say `raw_lengthscale`?

This is an important GPyTorch concept.

The actual parameter might need to obey constraints:

$$
\ell>0.
$$

Optimizers are easier to use if the underlying unconstrained parameter can range over all real numbers:

$$
z\in(-\infty,\infty).
$$

A transformation maps:

$$
z\rightarrow\ell>0.
$$

So you might see:

```text
raw_lengthscale
```

internally.

But the user-facing property:

```text
kernel.lengthscale
```

returns the constrained value.

Therefore:

```text
raw parameter
       │
       ▼
constraint transformation
       │
       ▼
actual parameter
```

### Why positivity constraints are needed

You cannot have:

$$
\ell=-0.5.
$$

A lengthscale must be positive.

Likewise:

$$
\sigma_f^2>0
$$

and:

$$
\sigma_n^2>0.
$$

GPyTorch handles these constraints internally.

This is one reason it is usually better to use the provided kernel/model APIs rather than manually optimizing these quantities.

###  `state_dict()`

A very useful API:

```python
state = model.state_dict()
```

This returns the model's parameter state.

For example:

```python
for key, value in model.state_dict().items():
    print(key, value)
```

This is useful for:

- debugging
- saving models
- comparing fitted models
- checkpointing
- restoring a model.

---

### Save the model state

You can do:

```python
torch.save(
    model.state_dict(),
    "gp_model.pt",
)
```

Then later:

```python
model.load_state_dict(
    torch.load("gp_model.pt")
)
```

However, when restoring a model, you need to reconstruct the same model architecture and data-dependent structure appropriately before loading the state.

We'll later cover robust model persistence.


## A complete parameter-inspection example

In [10]:
import torch

from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood


torch.set_default_dtype(torch.double)


train_X = torch.tensor([
    [0.00],
    [0.20],
    [0.40],
    [0.60],
    [0.80],
    [1.00],
])

train_Y = torch.tensor([
    [0.10],
    [0.85],
    [0.90],
    [0.20],
    [-0.80],
    [-0.10],
])


model = SingleTaskGP(
    train_X,
    train_Y,
)


mll = ExactMarginalLogLikelihood(
    model.likelihood,
    model,
)


fit_gpytorch_mll(mll)


print("MODEL")
print(model)

print("\nMEAN MODULE")
print(model.mean_module)

print("\nCOVARIANCE MODULE")
print(model.covar_module)

print("\nLIKELIHOOD")
print(model.likelihood)

print("\nPARAMETERS")
for name, param in model.named_parameters():
    print(name)
    print(param.detach())

MODEL
SingleTaskGP(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): GammaPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): MaternKernel(
      (lengthscale_prior): GammaPrior()
      (raw_lengthscale_constraint): Positive()
    )
    (outputscale_prior): GammaPrior()
    (raw_outputscale_constraint): Positive()
  )
)

MEAN MODULE
ConstantMean()

COVARIANCE MODULE
ScaleKernel(
  (base_kernel): MaternKernel(
    (lengthscale_prior): GammaPrior()
    (raw_lengthscale_constraint): Positive()
  )
  (outputscale_prior): GammaPrior()
  (raw_outputscale_constraint): Positive()
)

LIKELIHOOD
GaussianLikelihood(
  (noise_covar): HomoskedasticNoise(
    (noise_prior): GammaPrior()
    (raw_noise_constraint): GreaterThan(1.000E-04)
  )
)

PARAMETERS
likelihood.noise_covar.raw_noise
tensor([0.0328])
mean_module.raw_constant
tensor(0.1574)
covar_modul

/Users/benediktusmadika/Desktop/AI learning/botorch_sdl/lib/python3.9/site-packages/botorch/models/utils/assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([0.1917]), std = tensor([0.6344])). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)


## Don't just inspect parameters—inspect their scientific meaning

Suppose your fitted model gives approximately:

```text
lengthscale = 0.12
outputscale = 0.85
noise       = 0.015
```

Interpret them as:

### Lengthscale

The GP sees meaningful variation over relatively short distances in normalized input space.

### Outputscale

The underlying function has substantial variation.

### Noise

There is nonzero measurement variability.

But don't say:

> "The noise is 1.5% of the objective."

That interpretation depends on the model's outcome transformation and units.

This is why you must understand the transforms.

## Multiple input dimensions

Now let's construct a five-dimensional problem:

```python
train_X = torch.rand(30, 5)
train_Y = torch.randn(30, 1)
```

Then:

```python
model = SingleTaskGP(
    train_X,
    train_Y,
)
```

The kernel can learn dimension-specific behavior.

Conceptually:

$$
\boldsymbol{\ell}
=
[
\ell_1,\ell_2,\ell_3,\ell_4,\ell_5
].
$$

You can inspect:

```python
model.covar_module.base_kernel.lengthscale
```

and examine the dimensions.


### Why this is useful for electrolyte optimization

Suppose:

```text
dimension 1 = Li salt concentration
dimension 2 = EC
dimension 3 = DMC
dimension 4 = EMC
dimension 5 = temperature
```

The learned ARD structure gives you a first look at how rapidly the surrogate varies in each direction.

But because electrolyte variables have compositional constraints, interpretation must be done carefully.

For example:

$$
x_{EC}+x_{DMC}+x_{EMC}=1.
$$

The dimensions aren't independent.

Therefore, you should **not** interpret ARD lengthscales as if the formulation components could independently vary without constraint.

This becomes especially important later.

# `SingleTaskGP` is not the only GP model

You will eventually encounter:

```text
SingleTaskGP
ModelListGP
MultiTaskGP
SaasFullyBayesianSingleTaskGP
PairwiseGP
MixedSingleTaskGP
```

and others.

Their purposes differ.

For example:

```text
SingleTaskGP
```

is appropriate for a basic single-task objective.

Whereas:

```text
ModelListGP
```

can represent separate models for multiple outcomes.

And:

```text
MultiTaskGP
```

models correlations between multiple tasks.

We will cover these later.

---

## `SingleTaskGP` versus multiple outputs

Suppose:

$$
Y=
\begin{bmatrix}
\text{conductivity}\\
\text{capacity}\\
\text{cost}
\end{bmatrix}.
$$

You might have:

```text
train_Y.shape == [n, 3]
```

This does **not** automatically mean:

> "I now have a fully appropriate multi-objective GP."

There are several modeling choices.

For example:

```text
ModelListGP
```

can construct independent models:

$$
f_1(x),f_2(x),f_3(x).
$$

Other models can explicitly model output correlations.

We'll spend a dedicated lesson on this.

## A useful debugging checklist

In [11]:
print("X shape:", train_X.shape)
print("Y shape:", train_Y.shape)

print(model)

print("Mean:")
print(model.mean_module)

print("Kernel:")
print(model.covar_module)

print("Likelihood:")
print(model.likelihood)

print("Lengthscale:")
print(
    model.covar_module.base_kernel.lengthscale
)

print("Outputscale:")
print(
    model.covar_module.outputscale
)

print("Noise:")
print(
    model.likelihood.noise
)

X shape: torch.Size([6, 1])
Y shape: torch.Size([6, 1])
SingleTaskGP(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): GammaPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): MaternKernel(
      (lengthscale_prior): GammaPrior()
      (raw_lengthscale_constraint): Positive()
    )
    (outputscale_prior): GammaPrior()
    (raw_outputscale_constraint): Positive()
  )
)
Mean:
ConstantMean()
Kernel:
ScaleKernel(
  (base_kernel): MaternKernel(
    (lengthscale_prior): GammaPrior()
    (raw_lengthscale_constraint): Positive()
  )
  (outputscale_prior): GammaPrior()
  (raw_outputscale_constraint): Positive()
)
Likelihood:
GaussianLikelihood(
  (noise_covar): HomoskedasticNoise(
    (noise_prior): GammaPrior()
    (raw_noise_constraint): GreaterThan(1.000E-04)
  )
)
Lengthscale:
tensor([[0.3030]], grad_fn=<SoftplusBackward0>)
Outputscale:
tensor(

## Why `model.train()` and `model.eval()` exist

PyTorch models have modes.

You can call:

```python
model.train()
```

when training.

And:

```python
model.eval()
```

when evaluating.

This is especially important in probabilistic models because some components can behave differently during training/evaluation.

A safe conceptual pattern is:

```python
model.train()

fit_gpytorch_mll(mll)

model.eval()

posterior = model.posterior(X)
```

BoTorch's fitting routines handle much of the necessary model-mode management, but understanding these modes is important when you manually inspect or modify models.

### Why you shouldn't manually modify `raw_lengthscale`

Avoid doing something like:

```python
model.covar_module.base_kernel.raw_lengthscale = ...
```

unless you specifically understand GPyTorch's parameter/constraint system.

Prefer:

```python
model.covar_module.base_kernel.lengthscale = ...
```

when you actually need to set the constrained parameter.

And in normal BO, you generally let the marginal-likelihood optimizer learn it.

---

### What if the learned noise becomes extremely small?

Suppose:

$$
\sigma_n^2\approx10^{-8}.
$$

This could mean:

- your measurements are genuinely very precise,
- the data appear nearly deterministic,
- or the model is trying to explain variation through the latent function rather than noise.

You should not immediately conclude:

> "My experiment is noiseless."

For real SDL data, you should compare the fitted noise with:

- repeated measurements
- instrument precision
- cell-to-cell variation
- known experimental uncertainty.

---

### What if the lengthscale becomes extremely small?

Suppose:

$$
\ell_j\rightarrow0.
$$

Potential interpretations include:

- the objective genuinely changes rapidly along dimension $j$,
- sparse/insufficient data,
- overfitting,
- poor scaling,
- an inappropriate kernel,
- problematic experimental observations.

Again:

> GP hyperparameters are model parameters, not direct physical laws.

They are evidence that must be interpreted alongside the experiment.

## One complete inspection script

In [12]:
import torch

from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood


torch.set_default_dtype(torch.double)


# ------------------------------------------------------------
# Data
# ------------------------------------------------------------

train_X = torch.tensor([
    [0.00],
    [0.20],
    [0.40],
    [0.60],
    [0.80],
    [1.00],
])

train_Y = torch.tensor([
    [0.10],
    [0.85],
    [0.90],
    [0.20],
    [-0.80],
    [-0.10],
])


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = SingleTaskGP(
    train_X,
    train_Y,
)


# ------------------------------------------------------------
# Marginal likelihood
# ------------------------------------------------------------

mll = ExactMarginalLogLikelihood(
    model.likelihood,
    model,
)


# ------------------------------------------------------------
# Fit
# ------------------------------------------------------------

fit_gpytorch_mll(mll)


# ------------------------------------------------------------
# Inspect
# ------------------------------------------------------------

print("=" * 60)
print("MODEL")
print("=" * 60)

print(model)


print("\n" + "=" * 60)
print("MEAN MODULE")
print("=" * 60)

print(model.mean_module)


print("\n" + "=" * 60)
print("COVARIANCE MODULE")
print("=" * 60)

print(model.covar_module)


print("\n" + "=" * 60)
print("BASE KERNEL")
print("=" * 60)

print(model.covar_module.base_kernel)


print("\n" + "=" * 60)
print("LENGTHSCALE")
print("=" * 60)

print(
    model.covar_module.base_kernel.lengthscale
)


print("\n" + "=" * 60)
print("OUTPUTSCALE")
print("=" * 60)

print(
    model.covar_module.outputscale
)


print("\n" + "=" * 60)
print("LIKELIHOOD")
print("=" * 60)

print(model.likelihood)


print("\n" + "=" * 60)
print("NOISE")
print("=" * 60)

print(
    model.likelihood.noise
)


print("\n" + "=" * 60)
print("NAMED PARAMETERS")
print("=" * 60)

for name, parameter in model.named_parameters():
    print(
        f"{name}: "
        f"{parameter.detach()}"
    )


print("\n" + "=" * 60)
print("STATE DICT")
print("=" * 60)

for name, value in model.state_dict().items():
    print(
        f"{name}: "
        f"{value}"
    )

/Users/benediktusmadika/Desktop/AI learning/botorch_sdl/lib/python3.9/site-packages/botorch/models/utils/assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([0.1917]), std = tensor([0.6344])). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)


MODEL
SingleTaskGP(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): GammaPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): MaternKernel(
      (lengthscale_prior): GammaPrior()
      (raw_lengthscale_constraint): Positive()
    )
    (outputscale_prior): GammaPrior()
    (raw_outputscale_constraint): Positive()
  )
)

MEAN MODULE
ConstantMean()

COVARIANCE MODULE
ScaleKernel(
  (base_kernel): MaternKernel(
    (lengthscale_prior): GammaPrior()
    (raw_lengthscale_constraint): Positive()
  )
  (outputscale_prior): GammaPrior()
  (raw_outputscale_constraint): Positive()
)

BASE KERNEL
MaternKernel(
  (lengthscale_prior): GammaPrior()
  (raw_lengthscale_constraint): Positive()
)

LENGTHSCALE
tensor([[0.3030]], grad_fn=<SoftplusBackward0>)

OUTPUTSCALE
tensor(0.8459, grad_fn=<SoftplusBackward0>)

LIKELIHOOD
GaussianLikelihood(
  (noise_cova